# Notebook 04 — Feature Engineering

## Objectives
* Split data into train and test sets before any transformation
* Encode categorical variables using OrdinalEncoder
* Investigate numerical distributions and apply transformations if needed
* Save train and test sets to inputs/datasets/featured/

## Inputs
* `inputs/datasets/cleaned/marketing_campaign_cleaned.csv`

## Outputs
* `inputs/datasets/featured/X_train.csv`
* `inputs/datasets/featured/X_test.csv`
* `inputs/datasets/featured/y_train.csv`
* `inputs/datasets/featured/y_test.csv`

## Notes
* All transformers are fitted on the training set ONLY to prevent data leakage
* The train/test split uses stratify=y to preserve class imbalance ratio
* OrdinalEncoder is used over OneHotEncoder — low cardinality features,
  avoids dimensionality explosion, compatible with tree-based models

In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from feature_engine.encoding import OrdinalEncoder

os.chdir(os.path.dirname(os.getcwd()))
os.makedirs('inputs/datasets/featured', exist_ok=True)
print(f"Working directory: {os.getcwd()}")

Working directory: /workspaces/digital-marketing-conversion-predictor


---
## 1. Load Cleaned Data

In [2]:
df = pd.read_csv('inputs/datasets/cleaned/marketing_campaign_cleaned.csv')
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head()

Shape: (8000, 16)
Columns: ['Age', 'Gender', 'Income', 'CampaignChannel', 'CampaignType', 'AdSpend', 'ClickThroughRate', 'WebsiteVisits', 'PagesPerVisit', 'TimeOnSite', 'SocialShares', 'EmailOpens', 'EmailClicks', 'PreviousPurchases', 'LoyaltyPoints', 'Conversion']


,Age,Gender,Income,CampaignChannel,CampaignType,AdSpend,ClickThroughRate,WebsiteVisits,PagesPerVisit,TimeOnSite,SocialShares,EmailOpens,EmailClicks,PreviousPurchases,LoyaltyPoints,Conversion
0,56,Female,136912,Social Media,Awareness,6497.870068,0.043919,0,2.399017,7.396803,19,6,9,4,688,1
1,69,Male,41760,Email,Retention,3898.668606,0.155725,42,2.917138,5.352549,5,2,7,2,3459,1
2,46,Female,88456,PPC,Awareness,1546.429596,0.277490,2,8.223619,13.794901,0,11,2,8,2337,1
3,32,Female,44085,PPC,Conversion,539.525936,0.137611,47,4.540939,14.688363,89,2,2,0,2463,1
4,60,Female,83964,PPC,Conversion,1678.043573,0.252851,0,2.046847,13.993370,6,6,6,8,4345,1


---
## 2. Train-Test Split

We split BEFORE any transformation to prevent data leakage.
All encoders and transformers will be fitted on X_train only.

In [3]:
X = df.drop(columns=['Conversion'])
y = df['Conversion']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y  # preserves 87.65% / 12.35% ratio in both sets
)

print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")
print(f"y_train: {y_train.shape} | y_test: {y_test.shape}")
print(f"\nTrain class balance:\n{y_train.value_counts(normalize=True).round(3)}")
print(f"\nTest class balance:\n{y_test.value_counts(normalize=True).round(3)}")

X_train: (6400, 15) | X_test: (1600, 15)
y_train: (6400,) | y_test: (1600,)

Train class balance:
Conversion
1    0.877
0    0.123
Name: proportion, dtype: float64

Test class balance:
Conversion
1    0.876
0    0.124
Name: proportion, dtype: float64


---
## 3. Categorical Encoding

OrdinalEncoder assigns a numeric value to each category.
Fitted on X_train only, then applied to X_test.

In [4]:
cat_cols = X_train.select_dtypes(include='object').columns.tolist()
print(f"Categorical columns to encode: {cat_cols}")

encoder = OrdinalEncoder(encoding_method='arbitrary', variables=cat_cols)
encoder.fit(X_train)

X_train_enc = encoder.transform(X_train)
X_test_enc = encoder.transform(X_test)

print("\nEncoding mappings:")
for col, mapping in encoder.encoder_dict_.items():
    print(f"  {col}: {mapping}")

print(f"\nX_train_enc dtypes after encoding:")
print(X_train_enc.dtypes)

Categorical columns to encode: ['Gender', 'CampaignChannel', 'CampaignType']

Encoding mappings:
  Gender: {'Female': 0, 'Male': 1}
  CampaignChannel: {'Social Media': 0, 'Email': 1, 'SEO': 2, 'Referral': 3, 'PPC': 4}
  CampaignType: {'Retention': 0, 'Consideration': 1, 'Awareness': 2, 'Conversion': 3}

X_train_enc dtypes after encoding:
Age                    int64
Gender                 int64
Income                 int64
CampaignChannel        int64
CampaignType           int64
AdSpend              float64
ClickThroughRate     float64
WebsiteVisits          int64
PagesPerVisit        float64
TimeOnSite           float64
SocialShares           int64
EmailOpens             int64
EmailClicks            int64
PreviousPurchases      int64
LoyaltyPoints          int64
dtype: object


/workspaces/digital-marketing-conversion-predictor/.venv/lib/python3.14/site-packages/feature_engine/encoding/base_encoder.py:224: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  X[feature] = X[feature].map(self.encoder_dict_[feature])
/worksp

---
## 4. Save Featured Datasets

In [5]:
X_train_enc.to_csv('inputs/datasets/featured/X_train.csv', index=False)
X_test_enc.to_csv('inputs/datasets/featured/X_test.csv', index=False)
y_train.to_csv('inputs/datasets/featured/y_train.csv', index=False)
y_test.to_csv('inputs/datasets/featured/y_test.csv', index=False)

print("Saved:")
print(f"  X_train: {X_train_enc.shape}")
print(f"  X_test:  {X_test_enc.shape}")
print(f"  y_train: {y_train.shape}")
print(f"  y_test:  {y_test.shape}")

Saved:
  X_train: (6400, 15)
  X_test:  (1600, 15)
  y_train: (6400,)
  y_test:  (1600,)


---
## Summary

| Step | Method | Rationale |
|---|---|---|
| Train/test split | 80/20, stratified | Preserves class imbalance ratio |
| Categorical encoding | OrdinalEncoder | Low cardinality, tree-model compatible |
| Leakage prevention | Fit on X_train only | Encoder never sees test data |

Featured datasets saved to `inputs/datasets/featured/`.
Ready for Modelling notebook.